# Disappearance and improvement: did patients improve more when their seizure type left?

Runs on one **unit** (`UNIT`, the first code cell; see `cc_stats/units.py`): `types`, the seizure-type groups, or `codes`, the individual ILAE 2017 codes with at least 10 patients before surgery. The types notebook is the source; the codes one is generated from it by `analysis/unit_notebooks.py` (only `UNIT` and the notes differ), so edit the types notebook and regenerate. Below, "type" means a group of the unit, and the **core** groups are the unit's `pair_groups`: the five core seizure types, or the codes with at least 20 patients (focal awareness codes aside).

Among patients who had each seizure-type group before surgery: improvement when that type **left** after surgery vs when it **stayed**. Improvement among everyone with each type (with vs without it) is in `1_improvement/<unit>/Firth_results.ipynb` section 1; the table below repeats the overall rate for reference.

The rates and Wilson 95% CIs come from `improvement_rates.py`, which this notebook runs. It writes `improvement_rates.csv` to `results/<version>/2_disappearance/<unit>/`, for both outcomes. Every table and figure is shown for both outcomes, Engel then ILAE (see `cc_stats/outcomes.py`): improved = Engel I–III, or ILAE 1–4 (at least a 50% reduction in seizure days).

The stayed / left split needs post-op types as well, so its two groups don't always add up to the overall n.

In [ ]:
# The unit this notebook runs on: 'types' or 'codes' (cc_stats/units.py).
# Set here only; the codes notebook is generated with 'codes' (analysis/unit_notebooks.py).
UNIT = 'codes'

In [ ]:
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from cc_stats.outcomes import OUTCOMES, heading
from cc_stats.paths import ANALYSIS_CSV, DATASET, PROJECT_ROOT, results_dir
from cc_stats.plots import PAIR_A, PAIR_B, paired_rate_plot, save_figure
from cc_stats.units import unit as get_unit

analysis = pd.read_csv(ANALYSIS_CSV, index_col='subject_id')
unit = get_unit(UNIT, analysis)
# 'type' or 'code', for figure titles and labels
WORD = 'type' if UNIT == 'types' else 'code'
has_cols = [unit.pre(t) for t in unit.groups]
CORE = unit.pair_groups
CORE_NAME = 'core seizure types' if UNIT == 'types' else 'common seizure codes'
FOLDER = f'2_disappearance/{UNIT}'
# Figures grow with the number of rows (drawn for 10 groups, 5 core)
tall, tall_core = max(1, len(unit.groups) / 10), max(1, len(CORE) / 5)
print(f'dataset {DATASET}')

In [ ]:
RESULTS = results_dir(FOLDER)
run = subprocess.run([sys.executable, 'analysis/2_disappearance/improvement_rates.py', UNIT],
                     capture_output=True, text=True, cwd=PROJECT_ROOT)
if run.returncode != 0:
    raise RuntimeError(f'improvement_rates.py failed:\n{run.stderr}')
# The script computes both outcomes, one set of rows each.
all_rates = pd.read_csv(RESULTS / 'improvement_rates.csv')
rates = {outcome: all_rates.query('outcome == @outcome') for outcome in OUTCOMES}

In [ ]:
cohorts = {outcome: analysis.dropna(subset=[O['improved']] + has_cols) for outcome, O in OUTCOMES.items()}
for outcome, cohort in cohorts.items():
    print(f"{OUTCOMES[outcome]['name']}: {len(cohort)} patients with the outcome and pre-op seizure types recorded")

## Improvement when the type stayed vs left

In [ ]:
def fmt(row, group):
    n, k, r, lo, hi = (row[f'{group}_{c}'] for c in ('n', 'improved', 'rate', 'lo', 'hi'))
    return '' if n == 0 else f'{int(k)}/{int(n)}  {r:.0%} ({lo:.0%}-{hi:.0%})'

def rate_table(d):
    return pd.DataFrame({
        'type': d['type'],
        'all': [fmt(r, 'all') for _, r in d.iterrows()],
        'stayed': [fmt(r, 'stayed') for _, r in d.iterrows()],
        'left': [fmt(r, 'left') for _, r in d.iterrows()],
        'p (left vs stayed)': d['p'].map(lambda v: f'{v:.3f}'),
        'Holm p': d['p_holm'].map(lambda v: f'{v:.3f}'),
    }).style.hide(axis='index')

for outcome in OUTCOMES:
    heading(outcome)
    display(rate_table(rates[outcome]))

In [ ]:
for outcome, O in OUTCOMES.items():
    heading(outcome)
    d = rates[outcome].set_index('type').sort_values('all_rate')
    fig, ax = plt.subplots(figsize=(11, 7 * tall))
    paired_rate_plot(ax, d.index, top=(f'{WORD} left', d['left_improved'], d['left_n'], PAIR_A),
                     bottom=(f'{WORD} stayed', d['stayed_improved'], d['stayed_n'], PAIR_B),
                     p=d['p'], p_holm=d['p_holm'])
    ax.set_xlabel(f"Improvement, {O['improved_label']} (Wilson 95% CI)")
    ax.set_title(f"{O['name']} improvement: {WORD} stayed vs {WORD} left", loc='left', pad=28)
    ax.legend(loc='lower left', bbox_to_anchor=(0, 1.0), ncol=2, frameon=False, handletextpad=0.3)
    plt.tight_layout()
    save_figure(FOLDER, f'improvement_stayed_vs_left_all_groups_{outcome}')
    plt.show()

The same for the core groups only. The p and Holm p are those above (Holm across every group).

In [ ]:
for outcome, O in OUTCOMES.items():
    heading(outcome)
    d = rates[outcome].set_index('type').loc[CORE].sort_values('all_rate')
    fig, ax = plt.subplots(figsize=(10.5, 4.5 * tall_core))
    paired_rate_plot(ax, d.index, top=(f'{WORD} left', d['left_improved'], d['left_n'], PAIR_A),
                     bottom=(f'{WORD} stayed', d['stayed_improved'], d['stayed_n'], PAIR_B),
                     p=d['p'], p_holm=d['p_holm'])
    ax.set_xlabel(f"Improvement, {O['improved_label']} (Wilson 95% CI)")
    ax.set_title(f"{O['name']}: improvement when the {CORE_NAME.removesuffix('s')} stayed vs left", loc='left', pad=28)
    ax.legend(loc='lower left', bbox_to_anchor=(0, 1.0), ncol=2, frameon=False, handletextpad=0.3)
    plt.tight_layout()
    save_figure(FOLDER, f'improvement_stayed_vs_left_core_types_{outcome}')
    plt.show()

## Notes for interpretation

Checked against the v2 run on 2026-10-21 (14 codes; the core groups are the 7 codes with at least 20 patients, focal awareness codes aside).

- **Which codes go away** (`disappearance.ipynb`): most often atypical absence (11; 8 of 13, 62%), infantile spasms (38; 7 of 14) and generalized atonic (8; 16 of 33, 48%). Least often focal motor impaired awareness (15; 4 of 27), focal motor to BTC (24; 3 of 18), and at 19% each focal motor tonic (23; 5 of 26), generalized spasms (9; 4 of 21) and generalized tonic (4; 8 of 42). Not all focal codes are low: focal motor clonic (19; 3 of 8) and focal non-motor to BTC (33; 5 of 15) sit mid-table.
- **Left vs stayed:** no code's improvement differs by whether it left, on either outcome (every Holm p is 1.0; the smallest p is 0.18, infantile spasms on ILAE). The stayed / left split leaves 3 to 9 patients on one side for most codes, so this is a test with little power, not evidence of no effect.
- **Generalized myoclonic (5)**, the code that goes with improvement in `1_improvement/codes/`, goes with it whether it left or stayed: Engel 8/9 left and 14/14 stayed, ILAE 8/8 and 13/13. So its link to improvement does not come through the myoclonic seizures themselves going away.
- **Codes overlap** (a patient has a median of 4), and a focal seizure appears under both its awareness and its feature code, so the rows are not independent.